# Weather forecast


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import joblib

## Data preprocessing

In [ ]:
# Load the data
fcs = pd.read_csv("../data/external/fcast_data/fcast_SLOVENIA_1reg_2010.csv")

fcs.head()


In [ ]:
for column in fcs.columns:
    if column not in ['domain_meteosiId', 'valid_UTC', 'tsValid_issued']:
        print(f"{column}: {fcs[column].unique()}")


## 2010 - 2017

In [ ]:
# Dictionaries for converting the text values to numerical ones

# Cloud cover
nn_decodeText_dict = {
    'J' : 0,
    'PJ' : 1,
    'DO' : 2,
    'PO' : 3,
    'O' : 4,
    'M' : 5,
    '1/8 .. 2/8' : 0,
    '3/8 .. 4/8' : 1,
    '4/8 .. 5/8' : 2,
    '5/8 .. 6/8' : 3,
    '6/8 .. 7/8' : 4,
    '7/8 .. 8/8' : 5
}

# Rainfall
rr_decodeText_dict = {
    '0': 0,
    '0 .. 0.1': 1,
    '0.2 .. 0.4': 2,
    '[-] 0.0 .. 0.4 (<5)': 1,
    '0.4 ..0.8': 2,
    '[0] 0.4 ..1.2 (5 do 15)': 2,
    'mod': 2,
    '0.8 .. 2': 3,
    '[0] 1.2 .. 2.5 ( 15 do 30)': 3,
    '[+] 2.5 .. 4 ( 30 do 50)': 4,
    '[+] > 4 ( > 50)': 5
}

# Wind
ff_decodeText_dict = {
    '0': 0,
    '0 .. 1': 1,
    '1 .. 3': 1,
    '0 .. 2': 1,
    '< 2': 1,
    '2 .. 5': 2,
    '3 .. 5': 2,
    '5 .. 8': 3,
    '5 .. 10': 3,
    '10 .. 15': 4,
    '15 .. 20': 5,
    '20 .. 25': 6,
    '> 25': 7
}

mapping_dictionaries = {
    'nn_decodeText': nn_decodeText_dict,
    'rr_decodeText': rr_decodeText_dict,
    'ff_decodeText': ff_decodeText_dict
}

numerical_columns = ['tn', 'tx']


In [ ]:
fcs = pd.DataFrame()

# Iterate over all of the years
for year in range(2010, 2019):
    fcs_temp = pd.read_csv(f"../data/external/fcast_data/fcast_SLOVENIA_1reg_{year}.csv")
    fcs = pd.concat([fcs, fcs_temp])


In [ ]:
# Delete columns with more than 5% missing values
for column in fcs.columns:
    if fcs[column].isna().sum() / len(fcs) > 0.05:
        #delete the column
        fcs = fcs.drop(columns=[column])

# Delete the columns that contain the icon
for column in fcs.columns:
    if 'icon' in column:
        fcs = fcs.drop(columns=[column])

# Change valid_UTC and tsValid_issued to datetime and remove the time part
fcs['valid_UTC'] = pd.to_datetime(fcs['valid_UTC']).dt.date
fcs['tsValid_issued'] = pd.to_datetime(fcs['tsValid_issued']).dt.date


# Add new columns for the forecasts. Instead of having the forecasts for different horizons
# in separate rows, we want to have the forecasts for different horizons in separate columns.
# The new columns will be named "<column_name>_<horizon>"

# Create a new dataframe for the forecasts
fcs_new = pd.DataFrame()
fcs_new.index = pd.date_range(start='2010-01-01', end='2010-12-31', freq='D')

# Fill the new dataframe with the forecasts
for index,row in fcs.iterrows():
    horizon = (row['valid_UTC'] - row['tsValid_issued']).days
    for column in fcs.columns:
        if column not in ['domain_meteosiId', 'valid_UTC', 'tsValid_issued']:
            if pd.to_datetime(row['tsValid_issued']) >= pd.to_datetime('2010-01-01') and pd.to_datetime(row['tsValid_issued']) <= pd.to_datetime('2017-12-31'):
                fcs_new.loc[pd.to_datetime(row['tsValid_issued']), f"{column}_{horizon}"] = row[column]

# Change the values to numerical ones
# Either with the predefined dictionaries or by one-hot encoding
for column in fcs_new.columns:
    # Get the base column name by removing the horizon number at the end
    column_base = '_'.join(column.split('_')[:-1])

    if column_base in mapping_dictionaries.keys():
        # Convert the values to numerical ones using the predefined dictionaries
        fcs_new[column] = fcs_new[column].map(mapping_dictionaries[column_base])
    elif column_base in numerical_columns:
        continue
    else:
        # One-hot encode the values
        dummies = pd.get_dummies(fcs_new[column]).add_prefix(f"{column}_")
        # Drop original column and join the dummies
        fcs_new = fcs_new.drop(columns=[column])
        fcs_new = pd.concat([fcs_new, dummies], axis=1)


# Take care of the missing values
for column in fcs_new.columns:
    if column in numerical_columns:
        fcs_new[column] = fcs_new[column].interpolate(method='linear')
        fcs_new[column] = fcs_new[column].bfill()
    else:
        fcs_new[column] = fcs_new[column].ffill().bfill()

fcs_new


In [ ]:
# Check if we have to add something else to the dictionaries

forecasts = pd.DataFrame()

# Iterate over all of the years
for year in range(2010, 2018):
    fcs = pd.read_csv(f"../data/external/fcast_data/fcast_SLOVENIA_1reg_{year}.csv")
    forecasts = pd.concat([forecasts, fcs])

for column in forecasts.columns:
    # Get the base column name by removing the horizon number at the end
    column_base = '_'.join(column.split('_')[:-1])
    
    if column_base in mapping_dictionaries.keys():
        for value in forecasts[column].unique():
            if value not in mapping_dictionaries[column_base].keys():
                print(f"{column}: {value}")


In [ ]:
import matplotlib.pyplot as plt

# Plot tn_5, tn_4, ..., tn_0
plt.figure(figsize=(12, 6))
for i in range(6):
    plt.plot(fcs_new.index[-100:], fcs_new[f'tn_{i}'][-100:], label=f'tn_{i}')
plt.xlabel('Date')
plt.ylabel('Temperature (tn)')
plt.title('Temperature Forecasts for Different Horizons')
plt.legend()
plt.show()


In [ ]:
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-no-new-features.joblib')

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(aquifer_by_stations[85065]['date'], aquifer_by_stations[85065]['temperature_min'])
plt.xlabel('Date')
plt.ylabel('Temperature Min')
plt.title('Temperature Min')
plt.show()

In [ ]:
# Check the r2 scores between the predicted and real data
for horizon in range(0, 6):
    print(f"R2 score for horizon {horizon}: {r2_score(fcs_new[f'tn_{horizon}'], aquifer_by_stations[85065]['temperature_min'])}")

## Save the dataframe

In [ ]:
joblib.dump(fcs_new, '../data/interim/weather-forecast-slovenia-5-days.joblib')